<a href="https://colab.research.google.com/github/keondreblankson/project4/blob/main/KBlankson_Project4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project 4: Predictive Analysis with scikit-learn

Keondre Blankson  
IS 362

In my preprocessing assignment, I looked at odor and gill size to see how they relate to whether a mushroom is poisonous. Odor looked like the stronger feature in my tables. For this project, I wanted to see if that still held up when I trained a model and tested it on data the model had never seen.

## 1. Load and prepare the data

I used the same three columns and number codes from my preprocessing assignment. The target is `poisonous`, where 0 means edible and 1 means poisonous in the dataset. My two predictors are `odor` and `gill_size`.

[UCI Mushroom dataset and variable descriptions](https://archive.ics.uci.edu/dataset/73/mushroom)

In [1]:
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, recall_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/mushroom/agaricus-lepiota.data"
mushrooms = pd.read_csv(url, header=None, usecols=[0, 5, 8], na_values="?")
mushrooms.columns = ["poisonous", "odor", "gill_size"]

class_map = {"e": 0, "p": 1}
odor_map = {"a": 0, "l": 1, "c": 2, "y": 3, "f": 4, "m": 5, "n": 6, "p": 7, "s": 8}
gill_map = {"b": 0, "n": 1}

mushrooms["poisonous"] = mushrooms["poisonous"].map(class_map)
mushrooms["odor"] = mushrooms["odor"].map(odor_map)
mushrooms["gill_size"] = mushrooms["gill_size"].map(gill_map)

print("Missing values in the selected columns")
print(mushrooms.isna().sum().to_string())
assert not mushrooms.isna().any().any()
mushrooms = mushrooms.astype(int)

print("\nNumber of rows:", len(mushrooms))
print(mushrooms.head().to_string(index=False))

Missing values in the selected columns
poisonous    0
odor         0
gill_size    0

Number of rows: 8124
 poisonous  odor  gill_size
         1     7          1
         0     0          0
         0     1          0
         1     7          1
         0     6          0


## 2. Split the data and one-hot encode the predictors

I used 80% of the rows for training and 20% for testing. I used stratification to keep the proportion of edible and poisonous mushrooms close to the original dataset in both groups.

The odor and gill-size numbers stand for categories. For example, odor code 6 means no odor. I used `pd.get_dummies()` to make a separate 0 or 1 column for each category. I used the training columns to line up the test columns.

In [2]:
X = mushrooms[["odor", "gill_size"]]
y = mushrooms["poisonous"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

odor_train = pd.get_dummies(X_train["odor"], prefix="odor", dtype=int)
odor_test = pd.get_dummies(X_test["odor"], prefix="odor", dtype=int)
odor_test = odor_test.reindex(columns=odor_train.columns, fill_value=0)

gill_train = pd.get_dummies(X_train["gill_size"], prefix="gill", dtype=int)
gill_test = pd.get_dummies(X_test["gill_size"], prefix="gill", dtype=int)
gill_test = gill_test.reindex(columns=gill_train.columns, fill_value=0)

print("Training rows:", len(y_train))
print("Test rows:", len(y_test))
print("Odor columns:", len(odor_train.columns))
print("Gill-size columns:", len(gill_train.columns))

Training rows: 6499
Test rows: 1625
Odor columns: 9
Gill-size columns: 2


## 3. Train and compare the models

I trained two decision trees with the same settings. One used only odor and the other used only gill size. I tested both on the same rows. I also used a baseline that always predicts the most common class in the training data.

Accuracy is the percentage of all test cases the model got right. Poisonous recall is the percentage of actual poisonous cases it caught. I also counted how many poisonous examples each model predicted as edible.

In [3]:
odor_model = DecisionTreeClassifier(random_state=42)
odor_model.fit(odor_train, y_train)
odor_predictions = odor_model.predict(odor_test)

gill_model = DecisionTreeClassifier(random_state=42)
gill_model.fit(gill_train, y_train)
gill_predictions = gill_model.predict(gill_test)

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(odor_train, y_train)
baseline_predictions = baseline.predict(odor_test)

results = []
for name, predictions in [
    ("Odor", odor_predictions),
    ("Gill size", gill_predictions),
    ("Most frequent class", baseline_predictions),
]:
    matrix = confusion_matrix(y_test, predictions, labels=[0, 1])
    results.append({
        "Model": name,
        "Accuracy (%)": 100 * accuracy_score(y_test, predictions),
        "Poisonous recall (%)": 100 * recall_score(y_test, predictions),
        "Poisonous predicted edible": int(matrix[1, 0]),
    })

results = pd.DataFrame(results).set_index("Model")
print(results.round(2).to_string())

missed_odor_codes = X_test.loc[(y_test == 1) & (odor_predictions == 0), "odor"]
print("\nOdor codes for poisonous cases missed by the odor model")
print(missed_odor_codes.value_counts().sort_index().to_string())

                     Accuracy (%)  Poisonous recall (%)  Poisonous predicted edible
Model                                                                              
Odor                        98.58                 97.06                          23
Gill size                   77.35                 60.41                         310
Most frequent class         51.82                  0.00                         783

Odor codes for poisonous cases missed by the odor model
odor
6    23


## 4. Conclusion and next steps

Odor was the better predictor in my test. The odor model got **98.58%** accuracy, and the gill-size model got **77.35%**. That is a difference of about **21.2 percentage points**. Both models did better than the baseline, which got **51.82%** by predicting the most common class.

The odor model still predicted 23 poisonous cases as edible. All 23 had no odor. The gill-size model made that same type of mistake 310 times. This matches what I saw in the preprocessing assignment. The no-odor group contains both edible and poisonous cases, so using odor alone can still miss poisonous examples.

If I continued this project, I would test odor and gill size together to see if gill size helps with the no-odor cases. I would also try adding features like spore-print color and use stratified cross-validation to see how consistent the results are across different splits. I would compare poisonous recall as well as accuracy. These results apply to the UCI data and should never be used to decide if a wild mushroom is safe to eat.